# **Тестирование API-сервиса.**

В рамках данного задания вам предстоит **протестировать ваш API-сервис для сокращения ссылок** (ака ДЗ 3). Основная цель – обеспечить качественное тестовое покрытие кода с процентом **покрытия не менее `90%`**, а также проверить работоспособность и устойчивость сервиса под нагрузкой.


## **Типы тестов:**

### **Юнит-тесты**
- Проверка основных функций (особенно, если есть сложная логика или парсинг `JSON`).
- Работа механизма генерации уникальных коротких ссылок.
- Помните, что в случае со сложной логикой юнит-тесты позволяют, в первую очередь, понять по примерам как работает код.
- Не злоупотребляйте юнит-тестами в тех местах, где они не нужны.

### **Функциональные тесты**
- Тестирование API через `TestClient` FastAPI.
- Проверка всех CRUD-операций с короткими ссылками.
- Тестирование поведения при передаче невалидных данных.
- Проверка работы механизма перенаправления (`GET /{short_code}`).
- Подумайте о необходимости создания отдельной тестовой базы данных и её последующей очистке.

### **Нагрузочное тестирование**
- Использовать Locust или аналогичные инструменты.
- Тестировать массовое создание ссылок.
- Оценить влияние кэширования на производительность.
- Результаты можете представить в произвольной форме.

### **Технические требования:**
- Использовать `pytest` и `httpx` для API-тестов.
- Для мокирования зависимостей (например, БД, кэша) использовать `pytest-mock` или `unittest.mock`.
- Для нагрузки использовать `Locust` (или `k6`, к примеру).
- Тесты должны лежать в папке `tests/` на одном уровне с `src/`.
- Проверить процент покрытия с помощью `coverage run -m pytest tests`.

Выше дан общий пайплайн, но вы можете реализовать несколько по-другому, если вы так считаете нужным и это обусловлено вашим пониманием того как надо тестировать ваш же код.

## **Формат сдачи работы:**

**Работа сдается в AnyTask.**

Код тестов необходимо сдавать вместе с кодом из ДЗ 3. Также должен быть файл `html` (или аналог), который будет визуализировать тестовое покрытие, и текстовое описание с инструкцией как запустить тесты (может быть включено в `README.md`). Процент покрытия тестами должен быть доступен проверяющему без запуска кода.

Если вы выложили код на GitHub, то можете приложить ту же ссылку, что и для ДЗ 3, в форму сдачи для ДЗ 4. Тесты в репозитории начинают проверяться только тогда, когда вы сдали ДЗ 4 в AnyTask.

## **Критерии оценки:**

| Критерий | Описание | Балл |
| :- | :- | :-: |
| **Процент покрытия** | Покрытие кода тестами не менее `90%` | `4`|
| **Корректность юнит-тестов** | Логика тестов верная, тесты охватывают основные сценарии | `1` |
| **Функциональные тесты** | Тестируется вся основная логика API (CRUD, редирект) | `3` |
| **Нагрузочные тесты** | Тестируется производительность сервиса под нагрузкой | `2` |

Критерии могут незначительно варьироваться в зависимости от конкретного случая на усмотрение проверяющего в рамках разумного.

Бонусные баллы, как обычно, на усмотрение проверяющего. Итоговая оценка за задание не может быть более 12.

### **Полезные ссылки:**
📌 Тестирование FastAPI:
- [Extended Testing File](https://fastapi.tiangolo.com/tutorial/testing/#extended-testing-file)
- [Async Tests](https://fastapi.tiangolo.com/advanced/async-tests/)

📌 Locust для нагрузочного тестирования:
- [Документация Locust](https://locust.io/)

📌 Покрытие кода тестами в Python:
- [pytest-cov](https://pytest-cov.readthedocs.io/en/latest/)

In [1]:
import os
os.makedirs("tests", exist_ok=True)
print("/test successfully made.")

/test successfully made.


In [ ]:
%%writefile tests/conftest.py
import pytest
from fastapi.testclient import TestClient
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker


from main import app, get_db, Base


SQLALCHEMY_DATABASE_URL = "sqlite:///./test_shortener.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
TestingSessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

@pytest.fixture(scope="function")
def db_session():
    Base.metadata.create_all(bind=engine)
    session = TestingSessionLocal()
    yield session
    session.close()

    Base.metadata.drop_all(bind=engine)

@pytest.fixture(scope="function")
def client(db_session):

    def override_get_db():
        yield db_session
        
    app.dependency_overrides[get_db] = override_get_db
    with TestClient(app) as c:
        yield c

@pytest.fixture(autouse=True)
def mock_redis(mocker):
    mocker.patch("main.redis_client", None)

In [ ]:
%%writefile tests/test_unit.py
from main import generate_short_code

def test_generate_short_code_length():
    code = generate_short_code(6)
    assert len(code) == 6
    code_long = generate_short_code(10)
    assert len(code_long) == 10

def test_generate_short_code_uniqueness():
    code1 = generate_short_code()
    code2 = generate_short_code()
    assert code1 != code2

In [ ]:
%%writefile tests/test_api.py
def test_register_and_login(client):
    response = client.post("/register", json={"username": "testuser", "password": "pass"})
    assert response.status_code == 200
    assert "access_token" in response.json()

    response_dup = client.post("/register", json={"username": "testuser", "password": "123"})
    assert response_dup.status_code == 400

    response_login = client.post("/token", data={"username": "testuser", "password": "pass"})
    assert response_login.status_code == 200

def test_shorten_url(client):
    response = client.post("/links/shorten", json={"original_url": "https://fastapi.tiangolo.com/"})
    assert response.status_code == 200
    assert "short_code" in response.json()

def test_shorten_url_invalid_data(client):
    response = client.post("/links/shorten", json={"wrong_field": "data"})
    assert response.status_code == 422

def test_redirect_and_stats(client):
    client.post("/links/shorten", json={"original_url": "https://google.com", "custom_alias": "goog"})
    
    redirect_res = client.get("/goog", follow_redirects=False)
    assert redirect_res.status_code in [307, 302, 301]
    assert redirect_res.headers["location"] == "https://google.com"

    stats_res = client.get("/links/goog/stats")
    assert stats_res.status_code == 200
    assert stats_res.json()["clicks"] == 1

def test_delete_link(client):
    client.post("/register", json={"username": "del_user", "password": "123"})
    token = client.post("/token", data={"username": "del_user", "password": "123"}).json()["access_token"]
    headers = {"Authorization": f"Bearer {token}"}

    client.post("/links/shorten", json={"original_url": "https://test.com", "custom_alias": "todel"}, headers=headers)

    del_res = client.delete("/links/todel", headers=headers)
    assert del_res.status_code == 200

    assert client.get("/links/todel/stats").status_code == 404

In [ ]:
%%writefile locustfile.py
from locust import HttpUser, task, between
import random
import string

class ShortenerLoadTest(HttpUser):
    wait_time = between(1, 2)

    def on_start(self):
        res = self.client.post("/links/shorten", json={"original_url": "https://locust.io"})
        if res.status_code == 200:
            self.short_code = res.json().get("short_code")
        else:
            self.short_code = None

    @task(1)
    def test_create_link(self):
        rand_str = ''.join(random.choices(string.ascii_letters, k=5))
        self.client.post("/links/shorten", json={"original_url": f"https://example.com/{rand_str}"})

    @task(3)
    def test_redirect(self):
        if self.short_code:
            self.client.get(f"/{self.short_code}", name="/[short_code] redirect")
